In [30]:
import pandas as pd
import importlib

import plotly.express as px

import Irina.utility_functions as uf

In [79]:
importlib.reload(uf)

<module 'Irina.utility_functions' from '/Users/irinavorobeva/PycharmProjects/geoTopics/Irina/utility_functions.py'>

In [9]:
df_global = pd.read_csv(uf.PATH+"df_global_comparison_yearly.csv")

In [129]:
start_year = 1970
end_year = 2023

In [195]:
df_global_topics = (
    df_global
    .merge(uf.df_topics[["subfield_id", "subfield_name", "field_name", "domain_name"]].drop_duplicates(),
           on="subfield_id", how="left")
    .assign(
        probability_individual_change=lambda df: df.groupby("subfield_id").probability_individual.transform(lambda x: (x - x.iloc[0])),
        probability_individual_relative=lambda df: df.groupby("subfield_id").probability_individual.transform(lambda x: (x / x.iloc[0])),
    )
    # .query("subfield_id in @subfields_list")
)

In [125]:
df_prob_ind = (
    df_global
    .pivot(index="year", columns="subfield_id", values="probability_individual")
)

df_loss = (
    df_global
    .pivot(index="year", columns="subfield_id", values="counts_loss_pct")
)

In [82]:
uf.plotly_heatmap(
    df_prob_ind,
    x_labels=df_prob_ind.columns.astype(str),
    y_labels=df_prob_ind.index,
    x_type="subfield",
    z_min=0, z_max=0.02,
    line_height=10,
    colorscale="non-symmetric"
)

In [83]:
fig = uf.plotly_heatmap(
    df_loss,
    x_labels=df_prob_ind.columns.astype(str),
    y_labels=df_prob_ind.index,
    x_type="subfield",
    z_min=0, z_max=100,
    line_height=10,
    colorscale="non-symmetric",
    title="Global collaboration losses by subfields (%)"
)
fig.show()
# fig.write_image("../images/global_collaboration_losses_all_1970_2023.png",
#                 width=1200, height=600, scale=2)

In [102]:
year = 2023

fig = px.box(
    df_global_topics.query("year == @year"),
    x="domain_name",           # each domain = one box
    y="counts_loss_pct",       # values to summarize
    points="all",              # optional: show all points
    color="domain_name",       # color boxes by domain
    height=600,
    # title=f"Distribution of counts_loss_pct by domain in {year}",
    hover_data=["domain_name", "counts_loss_pct", "subfield_name"],
)

fig.update_layout(
    yaxis_title="Counts loss (%)",
    xaxis_title="Domain",
    showlegend=False
)

fig.show()
# fig.write_image("../images/global_collaboration_losses_by_domain_2023.png",
#                 width=1200, height=600, scale=2)

In [103]:
year = 2023
domain_name = "Social Sciences"

fig = px.box(
    df_global_topics.query("year == @year").query(f"domain_name == @domain_name"),
    x="field_name",           # each domain = one box
    y="counts_loss_pct",       # values to summarize
    points="all",              # optional: show all points
    color="field_name",       # color boxes by domain
    height=600,
    # title=f"Distribution of counts_loss_pct by field in {year} in {domain_name}",
    hover_data=["field_name", "counts_loss_pct", "subfield_name"],
)

fig.update_layout(
    yaxis_title="Counts loss (%)",
    xaxis_title="Domain",
    showlegend=False
)

fig.show()
# fig.write_image("../images/global_collaboration_losses_social_by_field_2023.png",
#                 width=1200, height=600, scale=2)

In [99]:
# year = 2023
# domain_name = "Social Sciences"
# px.bar(
#     (
#         df_global_topics
#         .assign(subfield=lambda df: df.subfield_id.astype(str))
#         .query("year == @year")
#         .query(f"domain_name == @domain_name")
#     ),
#     x="counts_loss_pct",
#     y="subfield_name",
#     color="field_name",
#     height=1000,
# )

In [40]:
df_global_topics.domain_name.drop_duplicates()

2          Life Sciences
56     Physical Sciences
83       Social Sciences
148      Health Sciences
Name: domain_name, dtype: object

In [202]:
fig = px.area(
    (
        df_global_topics
        .groupby(["domain_name", "year"], as_index=False)
        .probability_all
        .sum()
    ),
    x="year",
    y="probability_all",
    color="domain_name",
    labels={"probability_all": "Probability All", "year": "Year"},
    title="Articles distribution among science domains (including collaborations)"
)
fig.show()
# fig.write_image("../images/articles_by_domains_dynamics.png",
#                 width=1200, height=600, scale=2)

In [131]:
fig = px.area(
    (
        df_global_topics
        .groupby(["domain_name", "year"], as_index=False)
        .probability_individual
        .sum()
    ),
    x="year",
    y="probability_individual",
    color="domain_name",
    labels={"probability_individual": "Probability Individual", "year": "Year"},
    title="Articles distribution among science domains (excluding collaborations)"
)
fig.show()
# fig.write_image("../images/articles_by_domains_dynamics.png",
#                 width=1200, height=600, scale=2)

In [201]:
year = 2023

fig = px.box(
    df_global_topics.query("year == @year"),
    x="domain_name",           # each domain = one box
    y="probability_individual_change",       # values to summarize
    points="all",              # optional: show all points
    color="domain_name",       # color boxes by domain
    height=600,
    title=f"Probability {year} - Probability {start_year}",
    hover_data=["domain_name", "counts_loss_pct", "subfield_name", "subfield_id"],
)

fig.update_layout(
    # yaxis_title="Counts loss (%)",
    # xaxis_title="Domain",
    # showlegend=False
)

fig.show()
# fig.write_image("../images/probability_change_by_domain_2023.png",
#                 width=1200, height=600, scale=2)

In [155]:
subfield_top_10 = (
    df_global_topics
    .query("year == @year")
    .sort_values("probability_individual_change", ascending=False)
    .head(10)
    .subfield_id
    .to_list()
)
subfield_bottom_10 = (
    df_global_topics
    .query("year == @year")
    .sort_values("probability_individual_change", ascending=True)
    .head(10)
    .subfield_id
    .to_list()
)
subfield_stable_10 = (
    df_global_topics
    .query("year == @year")
    .assign(abs_change = lambda df: df.probability_individual_change.abs())
    .sort_values("abs_change", ascending=True)
    .head(10)
    .subfield_id
    .to_list()
)
subfields_list = [1306, 1313, 1111, 1312, 1110, 1105,
                  3304, 3312, 2002, 1202, 3314, 1211,
                  1702, 1710, 2208, 1607, 1605, 3107,
                  3600, 2730, 2739, 2712, 2737, 3404]

In [199]:
fig = px.line(
    df_global_topics.query("subfield_id == @subfield_top_10"),
    x="year",
    y="probability_individual",
    color="domain_name",      # now grouped by domain in text
    line_dash="subfield_name",
    hover_name="subfield_name",
    title="Probability growth: top 10",
)
fig.update_layout(
    legend=dict(
        title="Subfield:",
        orientation="h",   # horizontal
        yanchor="bottom",
        y=-0.5,           # distance below plot
        xanchor="center",
        x=0.5
    )
)

fig.show()

In [198]:
fig = px.line(
    df_global_topics.query("subfield_id == @subfield_bottom_10"),
    x="year",
    y="probability_individual",
    color="domain_name",      # now grouped by domain in text
    line_dash="subfield_name",
    hover_name="subfield_name",
    title="Probability growth: bottom 10",
)
fig.update_layout(
    legend=dict(
        title="Subfield:",
        orientation="h",   # horizontal
        yanchor="bottom",
        y=-0.5,           # distance below plot
        xanchor="center",
        x=0.5
    )
)

fig.show()

In [200]:
fig = px.line(
    df_global_topics.query("subfield_id == @subfield_stable_10"),
    x="year",
    y="probability_individual",
    color="domain_name",      # now grouped by domain in text
    line_dash="subfield_name",
    hover_name="subfield_name",
    title="Probability growth: stable 10",
)
fig.update_layout(
    legend=dict(
        title="Subfield:",
        orientation="h",   # horizontal
        yanchor="bottom",
        y=-0.5,           # distance below plot
        xanchor="center",
        x=0.5
    )
)

fig.show()

In [176]:
domain_name = "Life Sciences"
px.line(
    df_global_topics.query("subfield_id in @subfield_bottom_10"),
    x="year",
    y="probability_individual",
    color="domain_name",
    line_dash="subfield_name",
    hover_name="subfield_name",
    title="Bottom 10 by probability growth",
)

In [224]:
domain_name = "Health Sciences"
px.line(
    df_global_topics.query("subfield_id in @subfields_list and domain_name == @domain_name"),
    x="year",
    y="probability_individual",
    color="subfield_name",
    hover_name="subfield_name",
    title=domain_name,
)

In [238]:
import pandas as pd

def escape_latex(s):
    if isinstance(s, str):
        return (
            s.replace('&', '\\&')
             .replace('%', '\\%')
             .replace('$', '\\$')
             .replace('#', '\\#')
             .replace('_', '\\_')
             .replace('{', '\\{')
             .replace('}', '\\}')
             .replace('~', '\\textasciitilde{}')
             .replace('^', '\\^{}')
             .replace('\\', '\\textbackslash{}')
        )
    return s


In [239]:
(
    df_global_topics
    .query("year == @year")
    .sort_values("probability_individual", ascending=False)
    [["year", "subfield_name", "field_name", "probability_individual"]]
    .head(10)
)

,year,subfield_name,field_name,probability_individual
13552,2023,Sociology and Political Science,Social Sciences,0.035202
13425,2023,Electrical and Electronic Engineering,Engineering,0.034910
13362,2023,Molecular Biology,"Biochemistry, Genetics and Molecular Biology",0.032304
13544,2023,Education,Social Sciences,0.027781
13507,2023,Surgery,Medicine,0.023649
13387,2023,Artificial Intelligence,Computer Science,0.022269
13421,2023,Biomedical Engineering,Engineering,0.020484
13454,2023,Materials Chemistry,Materials Science,0.018805
13395,2023,Information Systems,Computer Science,0.018057
13427,2023,Mechanical Engineering,Engineering,0.016677


In [242]:
year = 2023
top_10_latex = (
    df_global_topics
    .query("year == @year")
    .sort_values("probability_individual", ascending=False)
    [["subfield_name", "field_name", "probability_individual"]]
    .head(10)
).applymap(escape_latex).to_latex(index=False)

top_10_latex

/var/folders/kq/3vlmcmh917x4z481rl4yj6t00000gn/T/ipykernel_6555/453121195.py:2: FutureWarning:

DataFrame.applymap has been deprecated. Use DataFrame.map instead.



,subfield_name,field_name,probability_individual
13552,Sociology and Political Science,Social Sciences,0.035202
13425,Electrical and Electronic Engineering,Engineering,0.034910
13362,Molecular Biology,"Biochemistry, Genetics and Molecular Biology",0.032304
13544,Education,Social Sciences,0.027781
13507,Surgery,Medicine,0.023649
13387,Artificial Intelligence,Computer Science,0.022269
13421,Biomedical Engineering,Engineering,0.020484
13454,Materials Chemistry,Materials Science,0.018805
13395,Information Systems,Computer Science,0.018057
13427,Mechanical Engineering,Engineering,0.016677


In [ ]:
top_10_latex

In [ ]:
(
    df_global_topics
    .groupby("year", group_keys=False)
    .apply(
        lambda g: g.nlargest(
            10,
            columns="probability_individual"
        )
    )
    [["year", "subfield_name", "field_name", "probability_individual"]]
)

In [222]:
# subfield_tree = uf.get_subfield_tree(uf.df_topics, [100, 10, 1])
# df_dist_global = uf.get_w1_distances(subfield_tree, df_prob_ind)

In [221]:
# df_prob_ind

In [220]:
# df_dist_global

In [223]:
# uf.plotly_heatmap(
#     df_dist_global,
#     x_labels=df_dist_global.columns,
#     y_labels=df_dist_global.index,
#     z_min=0, z_max=0.01,
#     colorscale="non_symmetric_difference"
# )